In [3]:
!pip install -q transformers datasets accelerate sentencepiece gradio

In [ ]:
#24BAD024

In [4]:
import torch
import numpy as np
import gradio as gr

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

print("Libraries imported successfully!")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Libraries imported successfully!
Device: cuda


In [5]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model.to(device)

print("Pre-trained FLAN-T5 model loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Pre-trained FLAN-T5 model loaded successfully!


In [6]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model.to(device)

print("Pre-trained FLAN-T5 model loaded successfully!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Pre-trained FLAN-T5 model loaded successfully!


In [7]:
questions = [
    "What is artificial intelligence?",
    "What is machine learning?",
    "What is deep learning?",
    "What is Python?",
    "What is data science?",
    "What is a large language model?",
    "What is natural language processing?",
    "What is a database?",
    "What is RAM?",
    "What is a CPU?",
    "What is a neural network?",
    "What is cloud computing?",
    "What is cybersecurity?",
    "What is an algorithm?",
    "What is a chatbot?",
    "What is supervised learning?",
    "What is unsupervised learning?",
    "What is a programming language?",
    "What is a computer network?",
    "What is data analysis?",
    "What is an operating system?",
    "What is a transformer model?",
    "What is generative AI?",
    "What is an API?",
    "What is a programming loop?",
    "How can students improve their study habits?",
    "What is the difference between RAM and ROM?",
    "What is the difference between AI and ML?",
    "What is a dataset?",
    "What is an epoch in machine learning?"
]

answers = [
    "Artificial Intelligence is a branch of computer science that enables machines to perform tasks that normally require human intelligence.",
    "Machine Learning is a part of AI that allows computers to learn from data and make predictions without being explicitly programmed for every task.",
    "Deep Learning is a part of machine learning that uses multi-layered neural networks to learn complex patterns.",
    "Python is a high-level programming language known for its simple syntax and wide range of applications.",
    "Data Science is the field of collecting, processing and analysing data to find useful information and patterns.",
    "A Large Language Model is an AI model trained on large amounts of text to understand and generate human-like language.",
    "Natural Language Processing is a branch of AI that helps computers understand and process human language.",
    "A database is an organized collection of data that can be stored, accessed and managed easily.",
    "RAM is temporary memory that stores data and programs currently being used by a computer.",
    "CPU stands for Central Processing Unit. It executes instructions and performs calculations in a computer.",
    "A neural network is a machine learning model inspired by the human brain that learns patterns using connected artificial neurons.",
    "Cloud computing provides computing services such as storage and servers over the internet.",
    "Cybersecurity is the practice of protecting computers, networks and data from digital attacks.",
    "An algorithm is a step-by-step set of instructions used to solve a problem.",
    "A chatbot is a software application that communicates with users through text or voice.",
    "Supervised learning is a machine learning method where a model learns from labelled training data.",
    "Unsupervised learning is a machine learning method where a model finds patterns in data without labelled answers.",
    "A programming language is a language used to write instructions that a computer can execute.",
    "A computer network is a group of connected devices that share data and resources.",
    "Data analysis is the process of examining data to find useful information and support decisions.",
    "An operating system is software that manages computer hardware and software resources.",
    "A transformer is a deep learning architecture that uses attention mechanisms to process information and understand relationships in data.",
    "Generative AI is artificial intelligence that can create new content such as text, images, audio and code.",
    "An API is a set of rules that allows different software applications to communicate with each other.",
    "A programming loop repeats a block of instructions until a condition is met or a fixed number of repetitions is completed.",
    "Students can improve their study habits by creating a timetable, setting goals, practising regularly and taking short breaks.",
    "RAM is temporary memory used while a computer is running, whereas ROM stores permanent instructions that are not normally changed.",
    "Artificial Intelligence is the broader field of making machines intelligent, while Machine Learning is a part of AI that learns from data.",
    "A dataset is a collection of related data used for analysis or training machine learning models.",
    "An epoch is one complete pass through the entire training dataset during model training."
]

data = Dataset.from_dict({
    "question": questions,
    "answer": answers
})

print("Dataset created successfully!")
print("Total questions:", len(data))

print(data[0])

Dataset created successfully!
Total questions: 30
{'question': 'What is artificial intelligence?', 'answer': 'Artificial Intelligence is a branch of computer science that enables machines to perform tasks that normally require human intelligence.'}


In [8]:
dataset = data.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print("Training samples:", len(train_dataset))
print("Testing samples:", len(eval_dataset))

Training samples: 24
Testing samples: 6


In [9]:
def preprocess_function(examples):

    inputs = [
        "Answer this educational question: " + q
        for q in examples["question"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["answer"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("Tokenization completed!")

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenization completed!


In [10]:
def preprocess_function(examples):

    inputs = [
        "Answer this educational question: " + q
        for q in examples["question"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["answer"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("Tokenization completed!")

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenization completed!


In [11]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

print("Data collator created!")

Data collator created!


In [12]:
training_args = Seq2SeqTrainingArguments(

    output_dir="./flan_t5_educational_results",

    num_train_epochs=10,

    per_device_train_batch_size=2,

    per_device_eval_batch_size=2,

    learning_rate=2e-5,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    save_total_limit=2,

    logging_steps=2,

    report_to="none",

    fp16=torch.cuda.is_available()
)

print("Fine-tuning parameters configured!")

Fine-tuning parameters configured!


In [13]:
trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_dataset["train"],

    eval_dataset=tokenized_dataset["test"],

    processing_class=tokenizer,

    data_collator=data_collator
)

print("Starting fine-tuning...")

trainer.train()

print("Fine-tuning completed successfully!")

Starting fine-tuning...


Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan
3,0.000000,nan
4,0.000000,nan
5,0.000000,nan
6,0.000000,nan
7,0.000000,nan
8,0.000000,nan
9,0.000000,nan
10,0.000000,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning completed successfully!


In [14]:
SAVE_PATH = "./fine_tuned_educational_chatbot"

trainer.save_model(SAVE_PATH)

tokenizer.save_pretrained(SAVE_PATH)

print("Fine-tuned model saved successfully!")
print("Model location:", SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved successfully!
Model location: ./fine_tuned_educational_chatbot


In [15]:
fine_tuned_model = AutoModelForSeq2SeqLM.from_pretrained(
    SAVE_PATH
)

fine_tuned_model.to(device)

fine_tuned_model.eval()

fine_tuned_tokenizer = AutoTokenizer.from_pretrained(
    SAVE_PATH
)

print("Fine-tuned model loaded successfully!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Fine-tuned model loaded successfully!


In [16]:
def generate_response(user_query):

    if not user_query or not user_query.strip():
        return "Please enter a question."

    prompt = (
        "Answer this educational question clearly and accurately "
        "using simple English: " + user_query
    )

    inputs = fine_tuned_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():

        output = fine_tuned_model.generate(
            **inputs,
            max_new_tokens=120,
            num_beams=4,
            do_sample=False,
            repetition_penalty=1.2,
            early_stopping=True
        )

    response = fine_tuned_tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return response.strip()


print(generate_response("What is artificial intelligence?"))

Artificial Intelligence (AI) is the development of artificial intelligence.


In [17]:
test_questions = [
    "What is artificial intelligence?",
    "Explain machine learning.",
    "What is cybersecurity?",
    "What is the difference between RAM and ROM?",
    "What is a neural network?"
]

for question in test_questions:

    print("Question:", question)

    print("Answer:", generate_response(question))

    print("-" * 60)

Question: What is artificial intelligence?
Answer: Artificial Intelligence (AI) is the development of artificial intelligence.
------------------------------------------------------------
Question: Explain machine learning.
Answer: Machine learning is the study of a computer program that learns a series of information from a single point of view.
------------------------------------------------------------
Question: What is cybersecurity?
Answer: cybersecurity is a security measure designed to protect the public from cyber attacks.
------------------------------------------------------------
Question: What is the difference between RAM and ROM?
Answer: RAM or ROM may refer to:
------------------------------------------------------------
Question: What is a neural network?
Answer: A neural network is a network of neurons in a human brain.
------------------------------------------------------------


In [18]:
demo = gr.Interface(

    fn=generate_response,

    inputs=gr.Textbox(
        lines=2,
        label="Ask Your Question",
        placeholder="Enter your academic question here..."
    ),

    outputs=gr.Textbox(
        lines=6,
        label="Chatbot Answer"
    ),

    title="Educational Institution Student Helper Chatbot",

    description=(
        "An AI-powered educational chatbot fine-tuned "
        "using FLAN-T5-Base to answer student questions."
    ),

    examples=[
        "What is artificial intelligence?",
        "Explain machine learning.",
        "What is Python?",
        "What is cybersecurity?",
        "How can students improve their study habits?"
    ]
)

print("Gradio interface created!")

Gradio interface created!


In [19]:
if 'demo' not in globals():
    import gradio as gr
    demo = gr.Interface(
        fn=generate_response,
        inputs=gr.Textbox(
            lines=2,
            label="Ask Your Question",
            placeholder="Enter your academic question here..."
        ),
        outputs=gr.Textbox(
            lines=6,
            label="Chatbot Answer"
        ),
        title="Educational Institution Student Helper Chatbot",
        description=(
            "An AI-powered educational chatbot fine-tuned "
            "using FLAN-T5-Base to answer student questions."
        ),
        examples=[
            "What is artificial intelligence?",
            "Explain machine learning.",
            "What is Python?",
            "What is cybersecurity?",
            "How can students improve their study habits?"
        ]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1142850f831fdd3178.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
